# 2장. 이 과제에 필요한 파이썬

| 항목 | 내용 |
|---|---|
| 예상 시간 | 3시간 |
| 선수 장 | 1장 |
| 이 장의 결과물 | 단어 빈도를 세고 정렬하는 함수, `fit`/`transform` 패턴 클래스 |

**학습 목표**
- 컴프리헨션, `enumerate`, `zip`으로 문서 리스트를 짧게 다룰 수 있다.
- `Counter`로 단어 빈도(TF)와 문서 빈도(DF)를 구분해서 셀 수 있다.
- 동점까지 고려해서 결과를 정렬할 수 있다.
- 과제의 `search` 함수 시그니처(타입 힌트)를 읽고 쓸 수 있다.
- 학습한 상태를 저장하는 `fit`/`transform` 패턴 클래스를 만들 수 있다.
- 실수(float)를 허용 오차로 비교하고, pytest로 함수를 검증할 수 있다.

이 장의 모든 예제는 미니 코퍼스를 씁니다. 아직 정제(8장)를 배우지 않았으므로, 여기서는 소문자로 바꾸고 끝의 문장부호만 떼는 간단한 방식으로 토큰을 만듭니다.

In [1]:
# 미니 코퍼스: 모든 장에서 함께 쓰는 문서 5개 (0장 0.5절 참고)
MINI_DOCS = [
    "The rocket launched into space!",
    "NASA launched a new space telescope.",
    "The pitcher threw a fast ball.",
    "Our team won the baseball game.",
    "Astronauts watched the baseball game in space.",
]
MINI_LABELS = ["space", "space", "baseball", "baseball", "space"]

# 간단한 토큰화: 소문자로 바꾸고, 끝의 . ! 를 떼고, 공백으로 자른다
TOKENS = [doc.lower().strip(".!").split() for doc in MINI_DOCS]
for doc_id, tokens in enumerate(TOKENS):
    print(doc_id, tokens)

0 ['the', 'rocket', 'launched', 'into', 'space']
1 ['nasa', 'launched', 'a', 'new', 'space', 'telescope']
2 ['the', 'pitcher', 'threw', 'a', 'fast', 'ball']
3 ['our', 'team', 'won', 'the', 'baseball', 'game']
4 ['astronauts', 'watched', 'the', 'baseball', 'game', 'in', 'space']


## 2.1 컴프리헨션, enumerate, zip

### 리스트 컴프리헨션
`[식 for 변수 in 반복가능 if 조건]` 형태로 for문 여러 줄을 한 줄로 씁니다.

In [2]:
# 문서별 토큰 수
lengths = [len(tokens) for tokens in TOKENS]
print("토큰 수:", lengths)

# 조건 붙이기: 불용어 빼기
STOP = {"the", "a", "into", "our", "in", "new"}
FILTERED = [[t for t in tokens if t not in STOP] for tokens in TOKENS]
print("불용어 제거:", FILTERED[0])

토큰 수: [5, 6, 6, 6, 7]
불용어 제거: ['rocket', 'launched', 'space']


### 이중 for: 2차원 리스트 펼치기
`[t for tokens in FILTERED for t in tokens]`는 **왼쪽 for가 바깥 반복문**입니다. 일반 for문으로 쓰면 아래와 순서가 같습니다.

```python
all_tokens = []
for tokens in FILTERED:      # 바깥
    for t in tokens:         # 안쪽
        all_tokens.append(t)
```

In [3]:
all_tokens = [t for tokens in FILTERED for t in tokens]
print("전체 토큰 수:", len(all_tokens))

# 집합 컴프리헨션: 중복 없는 단어 목록(어휘), 정렬까지
vocab = sorted({t for t in all_tokens})
print("어휘:", vocab)

# 딕셔너리 컴프리헨션: 단어 → 번호
word_to_index = {w: i for i, w in enumerate(vocab)}
print("space의 번호:", word_to_index["space"])

전체 토큰 수: 20
어휘: ['astronauts', 'ball', 'baseball', 'fast', 'game', 'launched', 'nasa', 'pitcher', 'rocket', 'space', 'team', 'telescope', 'threw', 'watched', 'won']
space의 번호: 9


`word_to_index`가 바로 11장에서 만들 **어휘 사전**입니다. 단어마다 열 번호를 정해 두면 문서를 숫자 배열로 바꿀 수 있습니다.

### enumerate와 zip
- `enumerate(리스트, start=0)`: (번호, 값) 쌍. 문서 ID나 순위를 붙일 때 씁니다.
- `zip(a, b)`: 두 리스트를 같은 위치끼리 묶습니다. 길이가 다르면 짧은 쪽에서 멈추므로, 실수를 잡고 싶으면 `strict=True`를 붙입니다.

In [4]:
for doc_id, (label, tokens) in enumerate(zip(MINI_LABELS, FILTERED, strict=True)):
    print(f"doc {doc_id} [{label}] {tokens}")

doc 0 [space] ['rocket', 'launched', 'space']
doc 1 [space] ['nasa', 'launched', 'space', 'telescope']
doc 2 [baseball] ['pitcher', 'threw', 'fast', 'ball']
doc 3 [baseball] ['team', 'won', 'baseball', 'game']
doc 4 [space] ['astronauts', 'watched', 'baseball', 'game', 'space']


> 컴프리헨션이 두 줄을 넘어갈 만큼 복잡해지면 일반 for문으로 쓰는 편이 읽기 쉽습니다.

## 2.2 Counter와 defaultdict

### Counter: 개수 세기
`collections.Counter`는 "값 → 개수" 딕셔너리입니다. 없는 키를 조회하면 에러 대신 0을 돌려줍니다.

In [5]:
from collections import Counter, defaultdict

counts = Counter(all_tokens)
print("가장 많이 나온 3개:", counts.most_common(3))
print("space 개수        :", counts["space"])
print("없는 단어          :", counts["moon"])  # KeyError 없이 0

가장 많이 나온 3개: [('space', 3), ('launched', 2), ('baseball', 2)]
space 개수        : 3
없는 단어          : 0


### TF와 DF를 구분해서 세기 (12장의 핵심)
- **TF(단어 빈도)**: 단어가 *몇 번* 나왔는가. 같은 문서에서 3번 나오면 3입니다.
- **DF(문서 빈도)**: 단어가 *몇 개 문서*에 나왔는가. 한 문서에서 여러 번 나와도 1로 셉니다.

DF는 문서마다 `set()`으로 중복을 없앤 뒤 세면 됩니다. 차이가 잘 보이도록 같은 단어가 반복되는 예제를 씁니다.

In [6]:
demo_docs = [
    ["space", "space", "rocket"],
    ["space", "game"],
    ["game", "game", "game"],
]

tf_total = Counter()
df = Counter()
for tokens in demo_docs:
    tf_total.update(tokens)     # 나온 횟수만큼 더한다
    df.update(set(tokens))      # 문서마다 한 번씩만 더한다

for word in ["space", "game", "rocket"]:
    print(f"{word:<7} 전체 등장 횟수 {tf_total[word]}  문서 빈도 {df[word]}")

space   전체 등장 횟수 3  문서 빈도 2
game    전체 등장 횟수 4  문서 빈도 2
rocket  전체 등장 횟수 1  문서 빈도 1


`game`은 모두 4번 나왔지만(2번째 문서에서 1번, 3번째 문서에서 3번) 문서로 따지면 2개뿐입니다. `space`는 3번 나왔고 문서도 2개입니다. 횟수와 문서 수는 다른 정보입니다.

### defaultdict: 처음 보는 키에 기본값 자동 생성
`defaultdict(list)`는 없는 키에 접근하면 빈 리스트를 만들어 줍니다. "단어 → 그 단어가 나온 문서 번호 목록"을 만들 때 편리합니다.

In [7]:
postings = defaultdict(list)
for doc_id, tokens in enumerate(FILTERED):
    for t in sorted(set(tokens)):
        postings[t].append(doc_id)   # 키가 없어도 빈 리스트가 자동으로 생긴다

for word in ["space", "game", "launched", "ball"]:
    print(f"{word:<9} → 문서 {postings[word]}")

space     → 문서 [0, 1, 4]
game      → 문서 [3, 4]
launched  → 문서 [0, 1]
ball      → 문서 [2]


일반 딕셔너리로 같은 일을 하려면 `postings.setdefault(t, []).append(doc_id)`나 `if t not in postings:` 검사가 필요합니다.

## 2.3 정렬

`sorted(반복가능, key=함수, reverse=False)`는 새 리스트를 돌려주고, `리스트.sort()`는 리스트 자체를 바꿉니다. `key`에는 "무엇을 기준으로 정렬할지"를 돌려주는 함수를 넣습니다. 한 번만 쓸 작은 함수는 `lambda`로 씁니다.

```python
lambda kv: kv[1]     # def f(kv): return kv[1] 과 같다
```

In [8]:
word_counts = Counter(all_tokens)

# 빈도만 기준으로 내림차순
by_count = sorted(word_counts.items(), key=lambda kv: kv[1], reverse=True)
print("빈도 기준      :", by_count[:6])

# 빈도 내림차순, 동점이면 단어 오름차순
by_count_then_word = sorted(word_counts.items(), key=lambda kv: (-kv[1], kv[0]))
print("빈도 → 알파벳  :", by_count_then_word[:6])

빈도 기준      : [('space', 3), ('launched', 2), ('baseball', 2), ('game', 2), ('rocket', 1), ('nasa', 1)]
빈도 → 알파벳  : [('space', 3), ('baseball', 2), ('game', 2), ('launched', 2), ('astronauts', 1), ('ball', 1)]


빈도만 기준으로 하면 동점인 단어의 순서는 **원래 리스트에 들어 있던 순서**를 따릅니다. 우연에 맡기지 않고 결과를 항상 같게 만들려면 `key`에 튜플 `(-빈도, 단어)`를 돌려줍니다. 튜플은 첫 값부터 비교하고, 같으면 다음 값을 비교합니다. 숫자에 `-`를 붙이면 그 기준만 내림차순이 됩니다.

### 상위 k개 뽑기
검색 결과의 상위 5개를 뽑는 것과 같은 패턴입니다.

In [9]:
scores = [0.12, 0.85, 0.40, 0.85, 0.05]   # 문서별 점수 (가짜)

top3 = sorted(enumerate(scores), key=lambda x: (-x[1], x[0]))[:3]
for rank, (doc_id, score) in enumerate(top3, start=1):
    print(f"{rank}위: doc {doc_id} ({score})")

1위: doc 1 (0.85)
2위: doc 3 (0.85)
3위: doc 2 (0.4)


doc 1과 doc 3이 0.85로 동점이지만 문서 번호가 작은 doc 1이 앞에 옵니다. 4장에서는 같은 일을 NumPy `argsort`로 훨씬 빠르게 합니다.

> **어휘 정렬:** `sorted(set(단어들))`은 알파벳 순입니다. sklearn의 `TfidfVectorizer`도 어휘를 알파벳 순으로 정렬하므로, 내 구현도 이렇게 해 두면 14장에서 결과를 열 단위로 바로 비교할 수 있습니다.

## 2.4 타입 힌트

타입 힌트는 변수와 함수가 어떤 자료형을 주고받는지 적어 두는 **설명**입니다.

| 표기 | 의미 |
|---|---|
| `x: int` | 정수 |
| `tokens: list[str]` | 문자열 리스트 |
| `vocab: dict[str, int]` | 키는 문자열, 값은 정수인 딕셔너리 |
| `result: tuple[float, int, str]` | (실수, 정수, 문자열) 3개짜리 튜플 |
| `x: int \| None` | 정수 또는 None |
| `def f(x: int) -> str:` | 정수를 받아 문자열을 돌려주는 함수 |

과제 문서의 시그니처 `search(query: str) -> List[(score, doc_id, text_snippet)]`를 정확한 파이썬 문법으로 쓰면 `search(query: str) -> list[tuple[float, int, str]]`입니다. 예전 코드에서 보이는 `List`, `Tuple`은 `typing` 모듈의 옛 표기이고, 파이썬 3.9부터는 소문자 `list`, `tuple`을 바로 씁니다.

아래는 시그니처만 과제와 같은 **아주 단순한 검색 함수**입니다. 점수는 "쿼리 단어 중 문서에 있는 비율"로 매깁니다. 15–16장에서 이 점수를 코사인 유사도로 바꿉니다.

In [10]:
SearchResult = tuple[float, int, str]   # 타입에 이름 붙이기 (타입 별칭)


def search(query: str, topk: int = 3) -> list[SearchResult]:
    query_words = set(query.lower().split())
    results: list[SearchResult] = []
    for doc_id, (text, tokens) in enumerate(zip(MINI_DOCS, FILTERED)):
        score = len(query_words & set(tokens)) / len(query_words)
        snippet = text[:30]
        results.append((score, doc_id, snippet))
    results.sort(key=lambda r: (-r[0], r[1]))
    return results[:topk]


for score, doc_id, snippet in search("space rocket"):
    print(f"[{score:.2f}] doc {doc_id}: {snippet}")

[1.00] doc 0: The rocket launched into space
[0.50] doc 1: NASA launched a new space tele
[0.50] doc 4: Astronauts watched the basebal


**주의:** 파이썬은 타입 힌트를 실행 중에 검사하지 않습니다. 힌트와 다른 값을 넣어도 그냥 실행됩니다. 힌트는 사람과 편집기(자동 완성, 경고)를 위한 것입니다.

In [11]:
def double(x: int) -> int:
    return x * 2


print(double(3))
print(double("ab"))   # 힌트는 int지만 에러 없이 "abab"가 나온다

6
abab


## 2.5 클래스로 상태 저장하기: fit/transform 패턴

TF-IDF는 두 단계로 쓰입니다.
1. **fit(학습)**: 학습 문서로 어휘와 IDF를 계산해서 **저장**합니다.
2. **transform(변환)**: 저장해 둔 어휘와 IDF로 *새 문서*(검색 쿼리, 테스트 문서)를 벡터로 바꿉니다.

학습 결과를 저장해 두어야 하므로 함수보다 **클래스**가 어울립니다. scikit-learn의 모든 변환기가 이 패턴을 따르고, 13장의 `NumpyTfidf`도 같은 모양으로 만듭니다.

규칙:
- `__init__`에는 **설정값**(하이퍼파라미터)만 저장합니다.
- `fit`에서 계산한 **학습 결과**는 이름 끝에 `_`를 붙입니다(`vocabulary_`). sklearn의 관례입니다.
- `fit`은 `self`를 돌려줘서 `WordCounter().fit(docs).transform(docs)`처럼 이어 쓸 수 있게 합니다.

In [12]:
class WordCounter:
    """어휘를 학습(fit)하고, 문서를 단어별 개수 리스트로 바꾼다(transform)."""

    def __init__(self, min_count: int = 1):
        self.min_count = min_count                       # 설정값
        self.vocabulary_: dict[str, int] | None = None   # 학습 결과 (fit 전에는 None)

    def fit(self, docs: list[list[str]]) -> "WordCounter":
        counts = Counter(t for tokens in docs for t in tokens)
        words = sorted(w for w, c in counts.items() if c >= self.min_count)
        self.vocabulary_ = {w: i for i, w in enumerate(words)}
        return self

    def transform(self, docs: list[list[str]]) -> list[list[int]]:
        if self.vocabulary_ is None:
            raise RuntimeError("transform 전에 fit을 먼저 호출하세요.")
        rows = []
        for tokens in docs:
            row = [0] * len(self.vocabulary_)
            for t in tokens:
                if t in self.vocabulary_:                # 학습 때 없던 단어는 무시
                    row[self.vocabulary_[t]] += 1
            rows.append(row)
        return rows

    def fit_transform(self, docs: list[list[str]]) -> list[list[int]]:
        return self.fit(docs).transform(docs)

앞의 4개 문서로만 학습하고, 학습에 쓰지 않은 doc 4를 변환해 봅니다.

In [13]:
counter = WordCounter().fit(FILTERED[:4])
print("학습한 어휘:", list(counter.vocabulary_))
print("doc 4 토큰 :", FILTERED[4])
print("doc 4 벡터 :", counter.transform([FILTERED[4]])[0])

try:
    WordCounter().transform(FILTERED)
except RuntimeError as e:
    print("\nfit 없이 transform:", e)

학습한 어휘: ['ball', 'baseball', 'fast', 'game', 'launched', 'nasa', 'pitcher', 'rocket', 'space', 'team', 'telescope', 'threw', 'won']
doc 4 토큰 : ['astronauts', 'watched', 'baseball', 'game', 'space']
doc 4 벡터 : [0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0]

fit 없이 transform: transform 전에 fit을 먼저 호출하세요.


doc 4의 `astronauts`, `watched`는 학습 어휘에 없어서 벡터에 반영되지 않았습니다. 학습 때 보지 못한 단어를 **OOV(Out-Of-Vocabulary)**라고 합니다. 검색 쿼리에 OOV만 있으면 벡터가 전부 0이 되는데, 이 경우의 처리를 15장에서 다룹니다.

> **왜 테스트 문서로 fit하면 안 될까?** 17장에서 다룰 "데이터 누수" 때문입니다. 평가용 문서의 정보(어휘, IDF)가 학습에 섞이면 실제보다 성능이 좋게 나옵니다.

## 2.6 파일 저장과 출력 포맷

### pathlib과 인코딩
`pathlib.Path`는 경로를 문자열이 아닌 객체로 다룹니다. `/`로 경로를 이어 붙이고, 파일 읽기와 쓰기도 메서드로 합니다. 한글이 있으면 **`encoding="utf-8"`을 꼭 적습니다**. 적지 않으면 Windows에서는 다른 인코딩(cp949)으로 저장되어 글자가 깨질 수 있습니다.

In [14]:
import json
import tempfile
from pathlib import Path

out_dir = Path(tempfile.mkdtemp()) / "outputs" / "en"   # 실습용 임시 폴더
out_dir.mkdir(parents=True, exist_ok=True)              # 중간 폴더까지 만들고, 이미 있어도 에러 없음

log_path = out_dir / "verify_log.txt"
log_path.write_text("검증 결과: PASS\n", encoding="utf-8")
print("저장 위치:", log_path.relative_to(out_dir.parent.parent))
print("읽은 내용:", log_path.read_text(encoding="utf-8").strip())

# JSON: 딕셔너리와 리스트를 파일로 저장하는 표준 형식
vocab_path = out_dir / "vocab.json"
vocab_path.write_text(json.dumps({"우주": 0, "야구": 1}, ensure_ascii=False), encoding="utf-8")
print("JSON     :", vocab_path.read_text(encoding="utf-8"))  # ensure_ascii=False여야 한글이 그대로 보인다

저장 위치: outputs/en/verify_log.txt
읽은 내용: 검증 결과: PASS
JSON     : {"우주": 0, "야구": 1}


### f-string 숫자 포맷
과제의 결과 예시처럼 로그를 깔끔하게 찍으려면 포맷 지정자를 씁니다.

| 지정자 | 예 | 결과 |
|---|---|---|
| `:.3f` | `f"{0.84712:.3f}"` | `0.847` (소수점 3자리) |
| `:.2e` | `f"{0.000000000832:.2e}"` | `8.32e-10` (지수 표기, 아주 작은 오차에) |
| `:>8` / `:<8` | `f"{'abc':>8}"` | 8칸 오른쪽 / 왼쪽 정렬 |
| `:,` | `f"{18000:,}"` | `18,000` |
| `=` | `f"{x=}"` | `x=3` (디버깅용) |

In [15]:
max_err, mean_err = 8.32e-10, 1.05e-12
shape = (500, 3247)

print("=== TF-IDF 구현 검증 결과 ===")
print(f"TF-IDF 행렬 shape: {shape}")
print(f"  - 최대 오차: {max_err:.2e}")
print(f"  - 평균 오차: {mean_err:.2e}")
print(f"  - 결과: {'PASS' if max_err < 1e-6 else 'FAIL'} (허용 오차 1e-6 이내)")
print()
for rank, (score, text) in enumerate([(0.8471, "CNN 기반 이미지 분류"), (0.7234, "딥러닝 객체 탐지")], start=1):
    print(f"{rank}. [{score:.3f}] {text}")

=== TF-IDF 구현 검증 결과 ===
TF-IDF 행렬 shape: (500, 3247)
  - 최대 오차: 8.32e-10
  - 평균 오차: 1.05e-12
  - 결과: PASS (허용 오차 1e-6 이내)

1. [0.847] CNN 기반 이미지 분류
2. [0.723] 딥러닝 객체 탐지


### 실수는 `==`로 비교하지 않는다
컴퓨터는 실수를 2진수로 저장하므로 아주 작은 오차가 생깁니다. 그래서 과제도 "sklearn과 **완전히 같은지**"가 아니라 "**오차가 1e-6 이내인지**"를 확인하라고 합니다.

In [16]:
a = 0.1 + 0.2
print(a)
print("a == 0.3            :", a == 0.3)
print("abs(a - 0.3) <= 1e-6:", abs(a - 0.3) <= 1e-6)

0.30000000000000004
a == 0.3            : False
abs(a - 0.3) <= 1e-6: True


## 2.7 pytest 맛보기

`assert 조건`은 조건이 거짓이면 `AssertionError`를 냅니다. **pytest**는 `test_`로 시작하는 파일에서 `test_`로 시작하는 함수를 찾아 모두 실행하고, 어떤 `assert`가 실패했는지 보여 줍니다.

예제로, 공백 기준 단어 수를 세는 함수에 버그를 하나 숨겨 두었습니다. `split(" ")`은 공백이 두 칸 연속이면 빈 문자열을 만듭니다.

In [17]:
import subprocess
import sys

test_dir = Path(tempfile.mkdtemp())
(test_dir / "textutil.py").write_text(
    "def count_words(text: str) -> int:\n"
    "    return len(text.split(' '))   # 버그: 연속 공백을 처리하지 못한다\n",
    encoding="utf-8",
)
(test_dir / "test_textutil.py").write_text(
    "from textutil import count_words\n"
    "\n"
    "def test_simple():\n"
    "    assert count_words('space rocket launch') == 3\n"
    "\n"
    "def test_double_space():\n"
    "    assert count_words('space  rocket') == 2\n",
    encoding="utf-8",
)


def run_pytest():
    result = subprocess.run(
        [sys.executable, "-m", "pytest", "-q", "--color=no", "-p", "no:cacheprovider"],
        cwd=test_dir, capture_output=True, text=True,
    )
    print(result.stdout.strip())


run_pytest()

.F                                                                       [100%]
=================================== FAILURES ===================================
______________________________ test_double_space _______________________________

    def test_double_space():
>       assert count_words('space  rocket') == 2
E       AssertionError: assert 3 == 2
E        +  where 3 = count_words('space  rocket')

test_textutil.py:7: AssertionError
=========================== short test summary info ============================
FAILED test_textutil.py::test_double_space - AssertionError: assert 3 == 2
1 failed, 1 passed in 0.01s


출력에서 `test_double_space`가 실패했고, `assert 3 == 2`처럼 **실제 값과 기대 값**을 보여 줍니다. 함수를 `text.split()`(인자 없음, 연속 공백을 하나로 취급)으로 고친 뒤 다시 실행합니다.

In [18]:
(test_dir / "textutil.py").write_text(
    "def count_words(text: str) -> int:\n"
    "    return len(text.split())\n",
    encoding="utf-8",
)
run_pytest()

..                                                                       [100%]
2 passed in 0.00s


실수 결과를 테스트할 때는 `assert result == pytest.approx(0.3)`처럼 `pytest.approx`를 쓰면 작은 오차를 허용합니다. 이 과제에서는 13장에서 손계산한 TF-IDF 값과 구현 결과를 비교하는 테스트를 작성합니다.

## ✍️ 실습 1: 단어 빈도 상위 k개

아래 문장들에서 가장 많이 나온 단어 상위 `k`개를 `(단어, 개수)` 리스트로 돌려주는 함수 `top_words(sentences, stopwords, k)`를 작성하세요.

- 소문자로 바꾸고, 공백으로 자르고, 각 단어 양끝의 `.,!?;` 문자를 떼어 냅니다 (`str.strip(".,!?;")`).
- 불용어와 빈 문자열은 제외합니다.
- 빈도 내림차순, 동점이면 단어 알파벳 오름차순으로 정렬합니다.

In [19]:
SENTENCES = [
    "The rocket reached orbit, and the crew cheered!",
    "NASA plans a new rocket launch next year.",
    "The pitcher threw the ball; the batter missed the ball.",
    "Baseball fans love a close game.",
    "The crew of the space station watched the game.",
    "A rocket engine test failed, so NASA delayed the launch.",
]
EX_STOP = {"the", "a", "and", "of", "so", "next", "new"}

In [20]:
# 여기에 작성하세요

**정답**

In [21]:
def top_words(sentences: list[str], stopwords: set[str], k: int) -> list[tuple[str, int]]:
    counts = Counter()
    for sentence in sentences:
        for raw in sentence.lower().split():
            word = raw.strip(".,!?;")
            if word and word not in stopwords:
                counts[word] += 1
    return sorted(counts.items(), key=lambda kv: (-kv[1], kv[0]))[:k]


print(top_words(SENTENCES, EX_STOP, 5))

[('rocket', 3), ('ball', 2), ('crew', 2), ('game', 2), ('launch', 2)]


## ✍️ 실습 2: 문서 빈도 상위 k개

같은 문장들에서, 단어가 **몇 개 문장에 나왔는지**(DF)를 기준으로 상위 `k`개를 돌려주는 `top_df(sentences, stopwords, k)`를 작성하세요. 실습 1의 결과와 비교해서 순위가 달라진 단어를 찾아보세요.

In [22]:
# 여기에 작성하세요

**정답**

In [23]:
def top_df(sentences: list[str], stopwords: set[str], k: int) -> list[tuple[str, int]]:
    df = Counter()
    for sentence in sentences:
        words = {raw.strip(".,!?;") for raw in sentence.lower().split()}
        df.update(w for w in words if w and w not in stopwords)
    return sorted(df.items(), key=lambda kv: (-kv[1], kv[0]))[:k]


print("TF 기준:", top_words(SENTENCES, EX_STOP, 5))
print("DF 기준:", top_df(SENTENCES, EX_STOP, 5))

TF 기준: [('rocket', 3), ('ball', 2), ('crew', 2), ('game', 2), ('launch', 2)]
DF 기준: [('rocket', 3), ('crew', 2), ('game', 2), ('launch', 2), ('nasa', 2)]


`ball`은 한 문장에서 두 번 나와 전체 빈도는 2지만, 문장 하나에만 있어서 DF는 1입니다. 그래서 DF 기준 상위 목록에서는 빠집니다. 이 차이가 12장 IDF의 출발점입니다.

## 🧐 퀴즈

**Q1.** `Counter(tokens)`와 `Counter(set(tokens))`는 각각 무엇을 세나요?
<details><summary>▶ 정답 보기</summary>

`Counter(tokens)`는 한 문서 안에서 단어가 나온 **횟수(TF)**를 셉니다. `Counter(set(tokens))`는 중복을 없앴으므로 모든 값이 1이고, 이것을 문서마다 `update`하면 단어가 나온 **문서 수(DF)**가 됩니다.
</details>

**Q2.** 점수 내림차순, 동점이면 문서 번호 오름차순으로 `(doc_id, score)` 리스트를 정렬하는 `key`는?
<details><summary>▶ 정답 보기</summary>

`key=lambda x: (-x[1], x[0])`. 튜플은 앞 값부터 비교하고, 점수에 `-`를 붙여 내림차순으로 만듭니다.
</details>

**Q3.** `0.1 + 0.2 == 0.3`이 `False`인 이유와, 두 실수를 비교하는 올바른 방법은?
<details><summary>▶ 정답 보기</summary>

실수는 2진수로 정확히 표현되지 않아 아주 작은 오차가 생깁니다. `abs(a - b) <= 허용오차`로 비교하거나, 배열은 `np.allclose`, 테스트에서는 `pytest.approx`를 씁니다.
</details>

**Q4.** `fit`과 `transform`을 하나의 함수로 합치지 않고 나누는 이유는?
<details><summary>▶ 정답 보기</summary>

학습 문서로 한 번 계산한 어휘와 IDF를 저장해 두고, 나중에 들어오는 쿼리나 테스트 문서를 **같은 기준**으로 변환해야 하기 때문입니다. 새 문서마다 다시 fit하면 어휘가 달라져 벡터끼리 비교할 수 없고, 테스트 데이터로 fit하면 데이터 누수가 생깁니다.
</details>

**Q5.** 함수에 `-> list[tuple[float, int, str]]` 힌트를 달았는데 실수로 문자열을 돌려주면 어떻게 되나요?
<details><summary>▶ 정답 보기</summary>

실행 중에는 아무 에러도 나지 않습니다. 파이썬은 타입 힌트를 강제하지 않습니다. 편집기 경고나 `mypy` 같은 별도 도구로만 잡을 수 있습니다.
</details>

## ✅ 체크포인트
- [ ] 2차원 리스트를 컴프리헨션 한 줄로 펼칠 수 있다.
- [ ] TF와 DF를 각각 `Counter`로 셀 수 있다.
- [ ] 동점까지 고려한 정렬 `key`를 쓸 수 있다.
- [ ] `fit`/`transform` 클래스에서 설정값과 학습 결과를 구분해 저장할 수 있다.
- [ ] 실수를 허용 오차로 비교할 수 있다.

## 🔗 과제 연결과 다음 장
- `Counter`, `word_to_index`, `fit`/`transform`은 11–13장 TF-IDF 구현의 뼈대가 됩니다.
- `SearchResult` 타입과 상위 k개 정렬은 16장 검색 함수의 시그니처로 그대로 쓰입니다.
- f-string 포맷과 허용 오차 비교는 14장 검증 로그에 쓰입니다.
- **3장. 수학 기초**에서 TF-IDF와 코사인 유사도 수식을 읽는 데 필요한 로그, 벡터, 내적을 배웁니다.